# Исследовательский анализ данных (EDA)
## Логистические перевозки и выявление аномалий

## 1. Загрузка библиотек и данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("Библиотеки загружены")

In [ ]:
# Загружаем данные
df = pd.read_csv('../data/processed/processed_shipments.csv', parse_dates=['date'])
print(f"Загружено {len(df)} записей")
df.head()

## 2. Общая информация о данных

In [ ]:
# Информация о датафрейме
df.info()

In [ ]:
# Статистическое описание
df.describe()

In [ ]:
# Проверка пропусков
df.isnull().sum()

## 3. Анализ распределений

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Detour ratio
axes[0, 0].hist(df['detour_ratio'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=1.5, color='red', linestyle='--', linewidth=2, label='Порог аномалии (1.5)')
axes[0, 0].set_xlabel('Коэффициент отклонения')
axes[0, 0].set_ylabel('Частота')
axes[0, 0].set_title('Распределение коэффициента отклонения')
axes[0, 0].legend()

# Скорость
axes[0, 1].hist(df['avg_speed_kph'], bins=40, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].axvline(x=60, color='red', linestyle='--', linewidth=2, label='Средняя скорость')
axes[0, 1].set_xlabel('Средняя скорость (км/ч)')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].set_title('Распределение скорости')
axes[0, 1].legend()

# Стоимость
axes[1, 0].hist(df['cost_rub'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_xlabel('Стоимость (руб)')
axes[1, 0].set_ylabel('Частота')
axes[1, 0].set_title('Распределение стоимости перевозок')

# Эффективность
axes[1, 1].hist(df['route_efficiency'], bins=40, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].axvline(x=0.85, color='green', linestyle='--', linewidth=2, label='Хорошо')
axes[1, 1].axvline(x=0.7, color='red', linestyle='--', linewidth=2, label='Плохо')
axes[1, 1].set_xlabel('Эффективность маршрута')
axes[1, 1].set_ylabel('Частота')
axes[1, 1].set_title('Распределение эффективности')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Анализ по городам

In [ ]:
# Статистика по городам отправления
city_stats = df.groupby('origin_city').agg({
    'shipment_id': 'count',
    'detour_ratio': 'mean',
    'avg_speed_kph': 'mean',
    'cost_rub': 'mean',
    'route_efficiency': 'mean'
}).round(2)

city_stats.columns = ['Кол-во', 'Ср.отклонение', 'Ср.скорость', 'Ср.стоимость', 'Эффективность']
city_stats.sort_values('Эффективность', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Эффективность по городам
efficiency_sorted = city_stats.sort_values('Эффективность', ascending=True)
colors = ['red' if x < 0.7 else 'orange' if x < 0.85 else 'green' for x in efficiency_sorted['Эффективность']]
efficiency_sorted['Эффективность'].plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_xlabel('Эффективность')
axes[0].set_title('Эффективность маршрутов по городам')
axes[0].axvline(x=0.85, color='orange', linestyle='--', alpha=0.7)
axes[0].axvline(x=0.7, color='red', linestyle='--', alpha=0.7)

# Скорость по городам
city_stats_speed = city_stats.sort_values('Ср.скорость', ascending=True)
city_stats_speed['Ср.скорость'].plot(kind='barh', ax=axes[1], color='lightblue')
axes[1].set_xlabel('Средняя скорость (км/ч)')
axes[1].set_title('Средняя скорость по городам')

plt.tight_layout()
plt.show()

## 5. Временной анализ

In [ ]:
# Анализ по дням недели
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = df.groupby('day_of_week').agg({
    'detour_ratio': 'mean',
    'avg_speed_kph': 'mean',
    'route_efficiency': 'mean'
}).reindex(day_order)

fig, ax = plt.subplots(figsize=(12, 6))
day_stats['route_efficiency'].plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
ax.axhline(y=0.85, color='orange', linestyle='--', linewidth=2, label='Хорошая эффективность')
ax.axhline(y=0.7, color='red', linestyle='--', linewidth=2, label='Критичная эффективность')
ax.set_xlabel('День недели')
ax.set_ylabel('Эффективность')
ax.set_title('Эффективность маршрутов по дням недели')
ax.legend()
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Динамика по месяцам
monthly_stats = df.groupby('month').agg({
    'detour_ratio': 'mean',
    'avg_speed_kph': 'mean',
    'cost_rub': 'mean'
})

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly_stats.index, monthly_stats['detour_ratio'], marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Месяц')
ax.set_ylabel('Коэффициент отклонения')
ax.set_title('Динамика отклонений по месяцам')
ax.grid(True, alpha=0.3)
plt.show()

## 6. Корреляционный анализ

In [ ]:
# Матрица корреляций
numeric_cols = ['optimal_distance_km', 'actual_distance_km', 'travel_time_hours', 
                'cost_rub', 'avg_speed_kph', 'detour_ratio', 'route_efficiency']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Корреляционная матрица признаков')
plt.show()

## 7. Выявление аномалий

In [ ]:
# IQR метод
Q1 = df['detour_ratio'].quantile(0.25)
Q3 = df['detour_ratio'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

anomalies_iqr = df[(df['detour_ratio'] < lower_bound) | (df['detour_ratio'] > upper_bound)]
print(f"IQR метод: найдено {len(anomalies_iqr)} аномалий ({len(anomalies_iqr)/len(df)*100:.1f}%)")

# Правила предметной области
anomalies_domain = df[(df['detour_ratio'] > 1.5) | (df['avg_speed_kph'] > 100) | (df['avg_speed_kph'] < 20)]
print(f"Domain rules: найдено {len(anomalies_domain)} аномалий ({len(anomalies_domain)/len(df)*100:.1f}%)")

# Комбинированный метод
df['is_anomaly'] = (df['detour_ratio'] > 1.5) | (df['avg_speed_kph'] > 100) | (df['avg_speed_kph'] < 20)
print(f"Комбинированный: найдено {df['is_anomaly'].sum()} аномалий ({df['is_anomaly'].sum()/len(df)*100:.1f}%)")

In [ ]:
# Топ-10 самых аномальных маршрутов
df[df['is_anomaly']].nlargest(10, 'detour_ratio')[['shipment_id', 'origin_city', 'destination_city', 
                                                      'detour_ratio', 'avg_speed_kph', 'cost_rub']]

## 8. Ключевые выводы

In [ ]:
print("=" * 60)
print("КЛЮЧЕВЫЕ ВЫВОДЫ EDA")
print("=" * 60)

print(f"\n1. Всего проанализировано перевозок: {len(df)}")
print(f"2. Выявлено аномальных маршрутов: {df['is_anomaly'].sum()} ({df['is_anomaly'].sum()/len(df)*100:.1f}%)")
print(f"3. Средний коэффициент отклонения: {df['detour_ratio'].mean():.2f}")
print(f"4. Максимальный коэффициент отклонения: {df['detour_ratio'].max():.2f}")
print(f"5. Средняя скорость: {df['avg_speed_kph'].mean():.1f} км/ч")
print(f"6. Общая стоимость перевозок: {df['cost_rub'].sum():,.0f} руб")

print("\nТОП-3 города с наибольшим количеством аномалий:")
top_cities = df[df['is_anomaly']]['origin_city'].value_counts().head(3)
for city, count in top_cities.items():
    print(f"   - {city}: {count} аномалий")

print("\nСамый проблемный день недели:")
worst_day = df[df['is_anomaly']]['day_of_week'].mode()[0]
print(f"   - {worst_day}")

print("\nРекомендации:")
print("   1. Оптимизировать маршруты из городов-лидеров по аномалиям")
print("   2. Усилить контроль в проблемные дни недели")
print("   3. Проверить маршруты с коэффициентом отклонения > 1.5")
print("   4. Внедрить автоматический мониторинг скорости")

print("\n" + "=" * 60)
print("EDA завершен!")
print("=" * 60)